In [1]:
import os
import pandas as pd


### Notebook Purpose
<b> This is notebook number 4.5 </b>

Allows us to double check training data/overlap data to make sure we don't screw something up when creating eval graphs

In [ ]:
training_data = pd.read_csv('../data/reviewed_data_5_13_24_training.csv')
holdout_data = pd.read_csv('../data/reviewed_data_5_13_24_holdout.csv')
training_data = pd.read_csv('../data/reviewed_data_5_13_24_training.csv')
justin_dataset = pd.read_csv('../data/dataset_may_models.csv').rename(columns = {'new_url':'download_url'})
seb_dataset = pd.read_csv('../data/13k_community_images_and_moderated_images.csv').rename(columns = {'new_url':'download_url'})
dataset = pd.read_csv('../data/hold_dataset_may_models_eval.csv').drop(columns=['Unnamed: 0'])

## First check uniqueness

In [13]:
def are_lists_wholly_unique(list1, list2):
    set1 = set(list1)
    set2 = set(list2)
    
    # Check if the intersection of the two sets is empty
    return set1.isdisjoint(set2)


In [4]:
print(
    are_lists_wholly_unique(training_data['download_url'].tolist(),
                            justin_dataset['download_url'].tolist())
                            ) 

False


In [27]:
len(justin_rs), len(seb_rs)

(410, 43)

In [23]:

print(
    are_lists_wholly_unique(training_data['download_url'].tolist(),
                           justin_dataset[justin_dataset['label'] == 'R']['download_url'].tolist())
                            ) 

False


In [5]:
print(
    are_lists_wholly_unique(training_data['download_url'].tolist(),
                            holdout_data['download_url'].tolist())
                            ) 

True


In [6]:
print(
    are_lists_wholly_unique(holdout_data['download_url'].tolist(),
                            justin_dataset['download_url'].tolist())
                            ) 

False


In [7]:
print(
    are_lists_wholly_unique(training_data['download_url'].tolist(),
                            dataset['download_url'].tolist())
                            ) 

True


In [8]:
print(
    are_lists_wholly_unique(holdout_data['download_url'].tolist(),
                            dataset['download_url'].tolist())
                            ) 

False


In [16]:
print(
    are_lists_wholly_unique(seb_dataset[seb_dataset['label'] == 'R']['download_url'].tolist(),
                            training_data['download_url'].tolist())
                            ) 

False


## Get class values we care about:

In [36]:
dataset.groupby('label').count()

,download_url,prompt,tags,prediction,model_preds
label,,,,,
PG,649,599,649,649,649
PG13,6158,5769,6158,6158,6158
X,2280,2160,2280,2280,2280
XXX,708,681,708,708,708


In [39]:
holdout_data.groupby('label')['download_url'].count()


label
PG       652
PG13    6174
X       2356
XXX      735
Name: download_url, dtype: int64

In [37]:
justin_dataset.groupby('label')['download_url'].count()

label
PG      1663
PG13    2173
R        479
X        188
XXX      194
Name: download_url, dtype: int64

In [40]:
dataset.columns, justin_dataset.columns, seb_dataset.columns


(Index(['download_url', 'prompt', 'tags', 'prediction', 'label', 'model_preds'], dtype='object'),
 Index(['download_url', 'prompt', 'tags', 'prediction', 'label', 'model_preds',
        'baseresNet18', 'baseresNet50', 'promptBert', 'promptTagBert',
        'resNet18CV', 'resNet50CV', 'roberta', '5_model_prediction'],
       dtype='object'),
 Index(['download_url', 'url', 'id', 'label', 'tags', 'prompt'], dtype='object'))

## Get from other datasets things we need

below we'll take R's from `seb` and `justin` datasets that `ARE NOT` in the training data, allowing us to build a better dataset for evaluation

In [52]:
tmp = None
for frame in [justin_dataset, seb_dataset]:
    rs = frame[(frame['label'] == 'R') &
               (~frame['download_url'].isin(training_data['download_url'])) &
               (~frame['download_url'].isin(dataset['download_url']))]
    if tmp is None:
        tmp = rs
    else:
        tmp = pd.concat([tmp, rs])

In [53]:
tmp.groupby('label')['download_url'].count()

label
R    360
Name: download_url, dtype: int64

In [54]:
dataset.groupby('label')['download_url'].count()

label
PG       649
PG13    6158
X       2280
XXX      708
Name: download_url, dtype: int64

In [55]:
## check we have no overlap with dataset OR with training data
print(
    are_lists_wholly_unique(tmp['download_url'].tolist(),
                            training_data['download_url'].tolist())
                            ) 

print(
    are_lists_wholly_unique(tmp['download_url'].tolist(),
                            dataset['download_url'].tolist())
                            ) 

True
True


In [56]:
pd.concat([dataset, tmp]).to_csv('../data/hold_dataset_june_models_eval.csv', index=False)